In [ ]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)
print("OpenCV version:", cv2.__version__)

In [ ]:
# Update this line to the path that actually contains your data
PROJECT_DIR = Path(r"C:\Users\macth\CPSC483_Labs\Emotion-Recognizer-main\Emotion-Recognizer-main")

DATASET_DIR = PROJECT_DIR / "data" / "DATASET"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "logs"

# This ensures the folders for your saved models and logs are created automatically
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)

# These should both return True now
print("Train exists:", TRAIN_DIR.exists())
print("Test exists:", TEST_DIR.exists())

if TRAIN_DIR.exists():
    print("Train folders:", sorted(os.listdir(TRAIN_DIR)))

In [ ]:
emotion_map = {
    "1": "surprise",
    "2": "fear",
    "3": "disgust",
    "4": "happy",
    "5": "sad",
    "6": "angry",
    "7": "neutral"
}

class_names = [
    "surprise",
    "fear",
    "disgust",
    "happy",
    "sad",
    "angry",
    "neutral"
]

print("Emotion classes:")
for key, value in emotion_map.items():
    print(key, "=", value)

In [ ]:
def count_images(folder):
    rows = []

    for class_folder in sorted(folder.iterdir()):
        if class_folder.is_dir():
            image_files = [
                f for f in class_folder.glob("*")
                if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
            ]

            rows.append({
                "folder_label": class_folder.name,
                "emotion": emotion_map.get(class_folder.name, "unknown"),
                "count": len(image_files)
            })

    return pd.DataFrame(rows)


train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("Training images:")
display(train_counts)

print("Testing images:")
display(test_counts)

print("Total train images:", train_counts["count"].sum())
print("Total test images:", test_counts["count"].sum())

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.6, 1.4], # Wider range to handle glare
    zoom_range=0.3,              # Zooms in on the mouth/nose area
    channel_shift_range=20.0,    # Randomly shifts colors to ignore glare
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=True,
    seed=SEED
)

test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

print("Class indices from Keras:")
print(train_gen.class_indices)

In [ ]:
# 1. Load the pre-trained 'Base' (The cerebral cortex model)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, # We remove the original 'head' to add our own
    weights="imagenet"
)

# Freeze the base so we don't 'break' its existing knowledge yet
base_model.trainable = False

# 2. Build the 'Head' (The part that learns YOUR specific emotions)
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.45)(x) # Prevents 'memorization' so it works on your face specifically
outputs = layers.Dense(7, activation="softmax")(x) # 7 neurons for 7 emotions

model = models.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), # Slow and steady learning
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"]
)

model.summary()

In [ ]:
index_to_folder = {v: k for k, v in train_gen.class_indices.items()}
index_to_emotion = {
    index: emotion_map[folder]
    for index, folder in index_to_folder.items()
}

emotion_labels = [
    index_to_emotion[i]
    for i in range(len(index_to_emotion))
]

print("Index to folder:")
print(index_to_folder)

print("Index to emotion:")
print(index_to_emotion)

print("Final emotion label order:")
print(emotion_labels)

In [ ]:
batch_images, batch_labels = next(train_gen)

plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)

    # MobileNetV2 preprocessing changes image range to [-1, 1].
    # Convert back to [0, 1] for display.
    img = (batch_images[i] + 1.0) / 2.0
    img = np.clip(img, 0, 1)

    label_index = int(batch_labels[i])
    label_name = emotion_labels[label_index]

    plt.imshow(img)
    plt.title(label_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y_train = train_gen.classes

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(np.unique(y_train), class_weights_array)
}

print("Class weights:")
print(class_weights)

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.45)(x)

outputs = layers.Dense(7, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
EPOCHS = 12

checkpoint_path = MODEL_DIR / "best_rafdb_emotion_model.keras"

callbacks = [
    ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
history_df = pd.DataFrame(history.history)

display(history_df)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 5

fine_tune_history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=fine_tune_epochs,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
if checkpoint_path.exists():
    model = tf.keras.models.load_model(checkpoint_path)
    print("Loaded best checkpoint:", checkpoint_path)

test_loss, test_accuracy = model.evaluate(test_gen)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

In [ ]:
test_gen.reset()

pred_probs = model.predict(test_gen)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_gen.classes

print("Classification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=emotion_labels
    )
)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=emotion_labels
)

fig, ax = plt.subplots(figsize=(9, 9))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("RAF-DB Confusion Matrix")
plt.show()

In [ ]:
final_model_path = MODEL_DIR / "rafdb_emotion_model.keras"
label_path = MODEL_DIR / "emotion_labels.txt"
label_json_path = MODEL_DIR / "emotion_labels.json"

model.save(final_model_path)

with open(label_path, "w", encoding="utf-8") as f:
    for label in emotion_labels:
        f.write(label + "\n")

with open(label_json_path, "w", encoding="utf-8") as f:
    json.dump(emotion_labels, f, indent=4)

print("Saved model to:", final_model_path)
print("Saved labels to:", label_path)
print("Saved labels JSON to:", label_json_path)

In [ ]:
def predict_single_image(image_path):
    img_bgr = cv2.imread(str(image_path))

    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))

    x = np.expand_dims(resized.astype(np.float32), axis=0)
    x = preprocess_input(x)

    probs = model.predict(x, verbose=0)[0]
    pred_index = int(np.argmax(probs))

    predicted_emotion = emotion_labels[pred_index]
    confidence = float(probs[pred_index])

    return predicted_emotion, confidence, img_rgb


sample_image = None

for folder in sorted(TEST_DIR.iterdir()):
    if folder.is_dir():
        image_list = [
            p for p in folder.glob("*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        ]

        if image_list:
            sample_image = image_list[0]
            break

emotion, confidence, img = predict_single_image(sample_image)

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f"Prediction: {emotion} ({confidence * 100:.1f}%)")
plt.axis("off")
plt.show()

print("Image:", sample_image)
print("Predicted emotion:", emotion)
print("Confidence:", confidence)

In [ ]:
import os

# Define the path to your training directory
train_dir = r"C:\Users\macth\CPSC483_Labs\Emotion-Recognizer-main\Emotion-Recognizer-main\data\DATASET\train"

# Add this line so the code knows what the numbers mean
emotion_map = {
    "1": "surprise",
    "2": "fear",
    "3": "disgust",
    "4": "happy",
    "5": "sad",
    "6": "angry",
    "7": "neutral"
}

print("File counts per category:")
print("-" * 30)

for folder in sorted(os.listdir(train_dir)):
    folder_path = os.path.join(train_dir, folder)
    if os.path.isdir(folder_path):
        count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
        emotion = emotion_map.get(folder, "Unknown")
        print(f"Folder {folder} ({emotion}): {count} images")

In [ ]:
# 1. Set the number of Epochs (how many times the model looks at the data)
EPOCHS = 15 

# 2. Add Callbacks (Safety features for training)
checkpoint_path = MODEL_DIR / "improved_emotion_model.keras"

callbacks = [
    # Saves the 'best' version of the model if accuracy improves
    tf.keras.callbacks.ModelCheckpoint(filepath=str(checkpoint_path), monitor="val_accuracy", save_best_only=True, verbose=1),
    # Stops training early if the model stops learning (prevents overfitting)
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1)
]

# 3. Start the Training (The 'Learning' Phase)
print("Starting training... This will take a few minutes.")
history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("Training complete! The best model has been saved to your models folder.")

In [ ]:
# Force the model to prioritize 'Disgust' (Index 2) 
# and 'Fear' (Index 1) over the common ones.
custom_class_weights = {
    0: 1.0, # Surprise
    1: 2.0, # Fear (Hard to see, so give it more weight)
    2: 2.5, # Disgust (Very hard to see, give it the most weight)
    3: 1.0, # Happy
    4: 0.8, # Sad (Model is too sensitive to this, so de-prioritize)
    5: 1.2, # Angry
    6: 0.8  # Neutral
}

# Use this in your model.fit()
model.fit(
    train_gen, 
    validation_data=test_gen, 
    epochs=5, 
    class_weight=custom_class_weights
)

In [ ]:
# 1. Unfreeze the base model
base_model.trainable = True

# 2. Re-freeze everything EXCEPT the last 50 layers
# This allows the model to fine-tune high-level facial features
for layer in base_model.layers[:-50]:
    layer.trainable = False

# 3. Use a VERY small learning rate
# We use 1e-5 (0.00001) so we don't 'destroy' the weights we already learned
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 4. Train for 5 more epochs
print("Fine-tuning started... this will make the model much more 'sensitive' to your face.")
model.fit(train_gen, validation_data=test_gen, epochs=5)

# 5. Save this as the 'Final' model
model.save(PROJECT_DIR / "models" / "final_emotion_model.keras")